In [1]:
from pathlib import Path

import pandas as pd

import numpy as np

from sentence_transformers import SentenceTransformer

from storm_rag.ingest import load_pdf_text, chunk_fixed, chunk_by_heading

In [2]:
PROJECT_ROOT = Path.cwd().parent  # adjust if your notebook isn't one level under project root

pdf_dir = PROJECT_ROOT / "data" / "docs"

pdf_paths = sorted(pdf_dir.glob("*.pdf"))

print(pdf_paths)

[PosixPath('/home/jtran/storm-events-rag/data/docs/ml.pdf'), PosixPath('/home/jtran/storm-events-rag/data/docs/mlfire1.pdf'), PosixPath('/home/jtran/storm-events-rag/data/docs/mlfire2.pdf'), PosixPath('/home/jtran/storm-events-rag/data/docs/pd01016005curr.pdf')]


In [3]:
texts = {p.stem: load_pdf_text(str(p)) for p in pdf_paths}

for name, t in texts.items():

    print(name, len(t), "chars")

ml 75444 chars
mlfire1 67286 chars
mlfire2 22776 chars
pd01016005curr 288796 chars


In [4]:
fixed_chunks = [

    {"doc": name, "strategy": "fixed", "chunk_id": i, "text": c}

    for name, t in texts.items()

    for i, c in enumerate(chunk_fixed(t))

]

heading_chunks = [

    {"doc": name, "strategy": "heading", "chunk_id": i, "text": c}

    for name, t in texts.items()

    for i, c in enumerate(chunk_by_heading(t))

]

chunks_df = pd.DataFrame(fixed_chunks + heading_chunks)

print(chunks_df.shape)

print(chunks_df["strategy"].value_counts())


(1056, 4)
strategy
fixed      651
heading    405
Name: count, dtype: int64


In [5]:
model = SentenceTransformer("BAAI/bge-small-en-v1.5")  # loads from local cache, no download

chunk_embeddings = model.encode(chunks_df["text"].tolist(), normalize_embeddings=True)

print(chunk_embeddings.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

(1056, 384)


In [6]:
def retrieve_top_k(query, strategy=None, k=3):

    mask = chunks_df["strategy"] == strategy if strategy else slice(None)

    df = chunks_df[mask].reset_index(drop=True)

    embs = chunk_embeddings[mask.values if strategy else mask]

    q_emb = model.encode([query], normalize_embeddings=True)

    scores = (embs @ q_emb.T).ravel()

    top_idx = np.argsort(-scores)[:k]

    for i in top_idx:

        row = df.iloc[i]

        print(round(float(scores[i]), 3), row["doc"], f"chunk {row['chunk_id']}")

        print(row["text"][:200], "\n")

In [7]:
questions = [

    "How do survey participants subjectively perceive and compare machine learning products to more traditionally derived products?",

    "How is deep learning used to predict fire occurrence in CONUS?",

"What are the reasons for performance improvements made possible by the deep learning model?"

]

for q in questions:

    print(f"\n########## {q} ##########")

    print("--- fixed ---")

    retrieve_top_k(q, strategy="fixed")

    print("--- heading ---")

    retrieve_top_k(q, strategy="heading")


########## How do survey participants subjectively perceive and compare machine learning products to more traditionally derived products? ##########
--- fixed ---
0.827 ml chunk 25
tive ques-
tions were included to help identify any differences in how
the respondents evaluate an ML product compared to more
traditional products. When designing this section of the sur-
vey, it was 

0.824 ml chunk 83
ly identical across the two methods, and any
variation fell well within the 95% conﬁdence interval. These re-
sults suggest that survey respondents on average do not con-
sciously evaluate ML-derived  

0.823 ml chunk 56
ts and
that these discrepancies may partially explain the anecdotally
perceived hesitancy of forecasters to adopt new ML products
operationally. Instead, the survey responses suggest that the re-
spon 

--- heading ---
0.793 ml chunk 4
0.70 0.77 10.10
Use by other experts in the ﬁeld 0.95 0.91 20.07
How closely the probabilistic output aligns with human-generated forecasts 